In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import dash
from dash import dcc, html
from dash.dependencies
import Input, Output
import plotly.express as px

print("Libraries installed successfully!")

df = pd.read_csv("failures_datan.csv", low_memory=False)

# Standardize column names
df.columns = df.columns.str.lower().str.strip()

Libraries installed successfully!


In [ ]:
df.head(2)

,bankno,bidcity,bidname,bidstate,brdate,bstatus,cert,chclass,chclass1,city,...,ptrdate,qbfasset,qbfdep,resdate,restype,restype1,savr,termi,uninsdep,url
0,NaN,0,0,0,NaN,NaN,NaN,NM,NM,EAST PEORIA,...,0,374.0,238.0,NaN,FAILURE,PO,FDIC,NaN,NaN,NaN
1,NaN,0,0,0,NaN,NaN,NaN,NM,NM,GRANTWOOD,...,0,2305.0,590.0,NaN,FAILURE,PO,FDIC,NaN,NaN,NaN


In [ ]:
#Convert Dates
df['faildate'] = pd.to_datetime(df['faildate'])

df['faildate'] = df['faildate'].dt.year

In [ ]:
# Numeric conversion 
numeric_cols = ['cost', 'qbfasset', 'qbfdep', 'uninsdep'] 
for col in numeric_cols: 
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
#Create Era
def classify_era(year):
    if year < 1980:
        return 'Historical'
    elif year < 2000:
        return 'S&L Crisis Era'
    elif year < 2015:
        return 'Modern Crisis Era'
    else:
        return 'Recent Era'

df['era'] = df['failyr'].apply(classify_era)

In [ ]:
#Show different Era's in different color
sns.scatterplot(
    data=df,
    x='QBFASSET',
    y='COST',
    hue='ERA'
)

In [ ]:
#To reduce skewness in cost and asset
df['log_cost'] = np.log1p(df['cost'])
df['log_asset'] = np.log1p(df['qbfasset'])

In [ ]:
#Statistics 
df[['cost', 'qbfasset', 'qbfdep', 'uninsdep']].describe()

In [ ]:
#Failure Cost by Bank Type
df.groupby('chclass')['cost'].mean()

In [ ]:
#Failure Cost by State
df.groupby('bidstate')['cost'].sum()

In [ ]:
#Failure by Time

df.groupby('failyr')['cost'].sum()

In [ ]:
#Average Cost by Bank Type
avg_cost = df.groupby('chclass')['cost'].mean()

avg_cost.plot(kind='bar')
plt.ylabel('Average Cost')
plt.title('Average Failure Cost by Bank Type')

In [ ]:
#Failure cost over years
yearly = df.groupby('failyr')['cost'].sum()

yearly.plot(kind='line')
plt.ylabel('Total Cost')
plt.title('Bank Failure Cost Over Time')

In [ ]:
#Distribution of Resolution Types
df['restype'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Distribution of Resolution Types')

In [ ]:
#Spread of Bank Asset
sns.boxplot(x=df['qbfasset'])
plt.title('Distribution of Bank Assets')


In [ ]:
#Scatter plot Asset vs Cost
plt.scatter(df['qbfasset'], df['cost'])
plt.xscale('log')
plt.yscale('log')
plt.xlabel('assets')
plt.ylabel('cost')
plt.title('Assets vs Failure Cost')

In [ ]:
#Scatter plot deposit vs cost
plt.scatter(df['qbfdep'], df['cost'])
plt.xscale('log')
plt.yscale('log')

In [ ]:
#Relationship Analysis
corr = df[['cost','qbfasset','qbfdep','uninsdep']].corr()
sns.heatmap(corr, annot=True)

In [ ]:
app = dash.Dash(__name__)
app.layout = html.Div([html.H1( "Bank Failure Cost Analysis Dashboard",
             style={'textAlign': 'center'} ),
# ========================= # FILTERS # ========================= 
             html.Div([ html.Label("Select Year"),
                        dcc.Dropdown( id='year_dropdown',
                        options=[ {'label': str(year), 'value': year} for year in sorted(df['failyr'].dropna().unique()) ],
                        value=df['failyr'].dropna().unique()[0],
                        clearable=False ) ],
                        style={'width': '30%', 'margin': 'auto'}),
                        html.Br(),
                        # ========================= # KPI CARDS # =========================
            html.Div(id='kpi_cards'),
            html.Br(), 
# ========================= # CHARTS # =========================
            dcc.Graph(id='bar_chart'),
            dcc.Graph(id='line_chart'),
            dcc.Graph(id='pie_chart'),
            dcc.Graph(id='box_plot'),
            dcc.Graph(id='scatter_plot'),
        ])
# ========================================= # CALLBACK # =========================================

@app.callback(
[ Output('kpi_cards', 'children'),
  Output('bar_chart', 'figure'),
  Output('line_chart', 'figure'), 
  Output('pie_chart', 'figure'),
  Output('box_plot', 'figure'),
  Output('scatter_plot', 'figure')
],

[Input('year_dropdown', 'value')] )

def update_dashboard(selected_year): 
    filtered_df = df[df['failyr'] == selected_year]
    # ===================================== # KPI VALUES # =====================================
    total_cost = filtered_df['cost'].sum()

    avg_cost = filtered_df['cost'].mean()

    total_banks = filtered_df['cert'].nunique()

    max_asset = filtered_df['qbfasset'].max()

    kpi_cards = html.Div([html.Div([ html.H3("Total Failure Cost"),
                           html.H4(f"${total_cost:,.0f}") ],
                           style={ 'border': '1px solid black',
                                   'padding': '20px',
                                   'width': '22%',
                                   'display': 'inline-block',
                                   'textAlign': 'center' }),
                           html.Div([html.H3("Average Cost"),
                                    html.H4(f"${avg_cost:,.0f}") ], 
                                    style={ 'border': '1px solid black',
                                            'padding': '20px',
                                            'width': '22%',
                                            'display': 'inline-block',
                                            'textAlign': 'center' }),
                          html.Div([ html.H3("Failed Banks"),
                                    html.H4(f"{total_banks}") ], 
                                    style={'border': '1px solid black', 
                                           'padding': '20px', 
                                           'width': '22%',
                                           'display': 'inline-block', 
                                           'textAlign': 'center' }),
                         html.Div([html.H3("Largest Asset"), 
                                    html.H4(f"${max_asset:,.0f}") ],
                                    style={ 'border': '1px solid black',
                                            'padding': '20px',
                                            'width': '22%',
                                            'display': 'inline-block',
                                            'textAlign': 'center' }), 
                                        ], 
                                            style={'display': 'flex', 'justifyContent': 'space-around'})

# ===================================== # BAR CHART # =====================================

bar_data = (filtered_df .groupby('chclass')['cost'] .mean() .reset_index() ) 
bar_fig = px.bar( bar_data, x='chclass', y='cost', title='Average Failure Cost by Bank Type' )

# ===================================== # LINE CHART# =====================================

yearly_data = ( df .groupby('failyr')['cost'] .sum() .reset_index() )
line_fig = px.line( yearly_data, x='failyr', y='cost', title='Bank Failure Cost Over Time' )

# ===================================== # PIE CHART# =====================================

pie_fig = px.pie( filtered_df, names='restype', title='Resolution Type Distribution' )

# ===================================== # BOX PLOT# =====================================

box_fig = px.box( filtered_df, y='log_cost', title='Spread of Failure Costs (Log Scale)' )

# ===================================== # SCATTER PLOT# =====================================

scatter_fig = px.scatter( filtered_df, x='log_asset', y='log_cost', color='chclass', hover_data=['name'], title='Assets vs Failure Cost' )

return ( kpi_cards, bar_fig, line_fig, pie_fig, box_fig, scatter_fig )

# RUN APP #
if __name__ == '__main__': 
    app.run_server(debug=True)


